# ¿Cómo Funciona el Algoritmo de Backpropagation?

Paso a paso de cómo funciona el algoritmo de retropropagación hacia atrás, que es una pieza fundamental de las Redes Neuronales Artificiales y muchas veces es considerado un "misterio".

Está desarrollado en "pseudo código", es decir, se explica como si estuvieramos con "papel y lápiz" desarrollando los cálculos. En un próximo blog discutiremos cómo implementarlo en Python usando Numpy.

De manera fundamental, el algoritmo actualizará los parámetros de la red retropropagando el error hacia las capas interiores de la misma. Este proceso se realiza "de derecha a izquierda", es decir, se parte primero estimando el efecto que los parámetros de las capas más cerca de la salida tienen en el output de la red, y con esa información se comienza a estimar progresivamente el efecto de los parámetros de las capas ocultas hasta llegar a las capas de entrada.

Para poder entender su funcionamiento, se recomienda tener conocimientos básicos de cálculo diferencial, esencialmente, de la regla de la cadena.

A continuación, discutimos en detalle este algoritmo.

## 1. Definiendo la Arquitectura de la Red

Supongamos una red simple con dos inputs i1 e i2, con una capa oculta con dos neuronas h1 y h2, y dos neuronas en la capa de salida o1 y o2:

![texto alternativo](https://matthewmazur.files.wordpress.com/2018/03/neural_network-7.png) Figura 1

En esta red, tanto las neuronas de la capa oculta, h1 y h2, y las de la capa de salida, o1 y o2 están siendo "activadas" ocupando la función logística (también conocida como sigmoide):

![texto alternativo](https://s0.wp.com/latex.php?latex=out_%7Bo1%7D+%3D+%5Cfrac%7B1%7D%7B1%2Be%5E%7B-net_%7Bo1%7D%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2)
Ecuación (1)

Supongamos ahora que queremos entrenar nuestra red con los siguientes valores:

- Cuando el input es **i1 = 0.05** queremos que nuestra salida sea **o1 = 0.01**
- Cuando el input es **i2 = 0.1** queremos que nuestra salida sea **o2 = 0.99**

En nuestra primera iteración inicializaremos los parámetros de la red aleatoriamente. Supongamos que obtenemos los siguientes valores para nuestros coeficientes w y b:

b1=0.35
b2=0.6

w1=0.15
w2=0.2
w3=0.25
w4=0.3
w5=0.4
w6=0.45
w7=0.5
w8=0.55

visualmente:

![texto alternativo](https://matthewmazur.files.wordpress.com/2018/03/neural_network-9.png) Figura 2

In [ ]:
# los inputs estan dados
i1=0.05
i2=0.1

# los outputs estan dados
o1=0.01
o2=0.99

# iniciamos los parámetros w y b aleatoriamente y obtenemos estos resultados:
b1=0.35
b2=0.6

w1=0.15
w2=0.2
w3=0.25
w4=0.3
w5=0.4
w6=0.45
w7=0.5
w8=0.55

# por supuesto en un código real tendriamos que inicializar los parámetros con algúna función aleatoria, por ejemplo, w_i=numpy.random.randn() o similar

## 2. Forward Pass

Como ya contamos con los pesos aleatorios iniciales, podemos hacer los cálculos correspondientes. Esto se conoce como *forward pass* o *feed forward*: haremos entrar los inputs a la red "de izquierda hacia la derecha", realizando todos los cálculos hasta obtener nuestra primera estimación de las salidas o1 y o2.

In [ ]:
# Propagando los datos hacia adelante ->

# Calculamos los valores netos de la neuronas de la capa oculta (netos, es decir, antes de aplicarles la función de activación logística)
# podemos ver que es solo una suma ponderada de los inputs a los que se les suma un término constante o "bias":

net_h1=i1*w1+i2*w2+b1
net_h2=i1*w3+i2*w4+b1

print('net_h1='+ str(net_h1))
print('net_h2='+str(net_h2))

net_h1=0.3775
net_h2=0.39249999999999996


In [ ]:
# Para calcular los valores finales o "output" de la neuronas, necesitamos aplicar la función de activación logística (u otra).

# primero la definimos ocupando la Ecuación 1:
import numpy as np
def logistic_func(net):
  out=1/(1+np.exp(-net))
  return out

# ahora la aplico a cada una de las neuronas de la capa oculta

out_h1=logistic_func(net_h1)
out_h2=logistic_func(net_h2)

print('out_h1='+ str(out_h1))
print('out_h2='+ str(out_h2))


out_h1=0.5932699921071872
out_h2=0.596884378259767


Dado que ya hemos calculado los valores de la capa oculta, podemos proceder a calcular los valores de las neuronas de la capa de salida:

In [ ]:
# primero calculamos el valor neto ocupando los pesos respectivos. notar que el input de las neuronas de la capa de salida o1 y o2,
#es el output de las neuronas de la capa oculta

net_o1=out_h1*w5+out_h2*w6+b2
net_o2=out_h1*w7+out_h2*w8+b2

print('net_o1='+ str(net_o1))
print('net_o2='+ str(net_o2))

net_o1=1.10590596705977
net_o2=1.2249214040964653


In [ ]:
# nuevamente, a estos valores netos le aplicamos la función de activación Ecuación 1.
out_o1=logistic_func(net_o1)
out_o2=logistic_func(net_o2)

print('out_o1='+ str(out_o1))
print('out_o2='+ str(out_o2))

out_o1=0.7513650695523157
out_o2=0.7729284653214625


Ya contamos con nuestra primera estimación para o1 y o2. Podemos comparar estos valores con respecto a los valores que deseamos que la red aprenda a replicar. La diferencia entre ellos es el **error** de nuestra red:

In [ ]:
# Calculamos los errores

e1=out_o1-o1
e2=out_o2-o2

print(e1,e2)

0.7413650695523157 -0.21707153467853746


Podemos observar que los valores predichos out_o1 y out_o2 están lejos de los valores buscados o1 y o2, por lo tanto, los errores son bastante grandes. Esto se debe a que los parámetros w y b escogidos todavía no han sido optimizados.

Para medir qué tan lejos estamos de los valores deseados, ocupamos una **función de costo** o **función de pérdida** (*Cost function / Loss Function*). En este ejemplo implementaremos una de las más utilizadas: el **Error Cuadrático** (*Squared Error*).

Se calcula simplemente como la diferencia al cuadrado, a veces multiplicada por la constante ½, para facilitar las derivadas que ocuparemos más adelante:

![texto alternativo](https://s0.wp.com/latex.php?latex=E_%7Btotal%7D+%3D+%5Csum+%5Cfrac%7B1%7D%7B2%7D%28target+-+output%29%5E%7B2%7D&bg=ffffff&fg=404040&s=0&zoom=2)
Ecuación (2)

In [ ]:
loss_e1=0.5*(e1**2)
loss_e2=0.5*(e2**2)

print(loss_e1,loss_e2)

0.274811083176155 0.023560025583847746


El error total de la red es la suma de los errores al cuadrado cometidos en cada predicción:

![texto alternativo](https://s0.wp.com/latex.php?latex=E_%7Btotal%7D+%3D+E_%7Bo1%7D+%2B+E_%7Bo2%7D+%3D+0.274811083+%2B+0.023560026+%3D+0.298371109&bg=ffffff&fg=404040&s=0&zoom=2)

In [ ]:
total_error=loss_e1+loss_e2
print(total_error)

0.2983711087600027


## 3. Backward Pass — Optimizando los Parámetros

Sabemos que los parámetros w y b escogidos inicialmente no son óptimos, por lo que ahora queremos optimizarlos usando **gradiente descendente**. La idea es estimar el efecto que cada parámetro tiene sobre el error total. Si sabemos que incrementar w5 reducirá el error total, entonces en la próxima iteración usaremos un valor w5_actualizado > w5_anterior.

Dado que la red tiene muchas capas y neuronas, el procedimiento consiste en ir estimando cómo afecta cada parámetro de manera "encadenada": recorremos la red desde la salida hacia la entrada (de derecha a izquierda), evaluando el efecto conjunto de cada sub-parte.

### 3.1 Efecto de los Parámetros de la Capa de Salida en el Error Total

Primero estimaremos el efecto de los parámetros w5, w6, w7, w8 en las neuronas de la capa de salida (o1 y o2) y, por consecuencia, en los errores e1 y e2 y en el error total.

Por ejemplo, ¿cuánto cambia el error total cuando cambia el parámetro w5? Observe la Figura 3:

![texto alternativo](https://matthewmazur.files.wordpress.com/2018/03/output_1_backprop-4.png) Figura 3

Vemos que se produce una "cadena": el error total depende de Eo1 y Eo2, y estos del output de las neuronas o1 y o2. A su vez, el output de o1 depende del valor neto de o1, el cual depende finalmente de w5 y del output de h1. Como E_o2 no depende de w5 ni directa ni indirectamente, podemos omitirlo.

Podemos expresar esta relación matemáticamente como una derivada parcial:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+w_%7B5%7D%7D+%3D+%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+out_%7Bo1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+out_%7Bo1%7D%7D%7B%5Cpartial+net_%7Bo1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+net_%7Bo1%7D%7D%7B%5Cpartial+w_%7B5%7D%7D&bg=ffffff&fg=404040&s=2&zoom=2) Ecuación (3)

Al conjunto de derivadas que representan cómo se ve afectado el error por cada w y b se le conoce como **Gradiente**. Note que el efecto de w5 (Ecuación 3) puede descomponerse en los tres componentes de la "cadena" descrita en la Figura 3.

#### 3.1.1 Componente 1: cómo cambia el error total cuando cambia out_o1

Para estimar este efecto, derivamos el error total (Ecuación 2) respecto al output de la neurona o1:

![texto alternativo](https://s0.wp.com/latex.php?latex=E_%7Btotal%7D+%3D+%5Cfrac%7B1%7D%7B2%7D%28target_%7Bo1%7D+-+out_%7Bo1%7D%29%5E%7B2%7D+%2B+%5Cfrac%7B1%7D%7B2%7D%28target_%7Bo2%7D+-+out_%7Bo2%7D%29%5E%7B2%7D&bg=ffffff&fg=404040&s=0&zoom=2)

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+out_%7Bo1%7D%7D+%3D+2+%2A+%5Cfrac%7B1%7D%7B2%7D%28target_%7Bo1%7D+-+out_%7Bo1%7D%29%5E%7B2+-+1%7D+%2A+-1+%2B+0&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación 4

Dado que conocemos el valor objetivo o1 (*target*) y el valor estimado por la red (out_o1), podemos calcular este efecto:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+out_%7Bo1%7D%7D+%3D+-%28target_%7Bo1%7D+-+out_%7Bo1%7D%29+%3D+-%280.01+-+0.75136507%29+%3D+0.74136507&bg=ffffff&fg=404040&s=0&zoom=2)

In [ ]:
# numéricamente:

delta_e_wr_delta_out_o1=-(o1-out_o1)
print(delta_e_wr_delta_out_o1)

0.7413650695523157


Notar que el resultado es simplemente el **Error o1**.

In [ ]:
print(e1)

0.7413650695523157


#### 3.1.2 Componente 2: cómo cambia el output de o1 respecto a su entrada neta

Es equivalente a encontrar la derivada de la función de activación (logística) respecto a su input. Recordando la Ecuación (1):

![texto alternativo](https://s0.wp.com/latex.php?latex=out_%7Bo1%7D+%3D+%5Cfrac%7B1%7D%7B1%2Be%5E%7B-net_%7Bo1%7D%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2)

![texto alternativo](https://wikimedia.org/api/rest_v1/media/math/render/svg/1bb35c81bc043a65aef427a578c207f847472547) función logística o sigmoide

Su derivada respecto a su entrada neta es:

![texto alternativo](https://wikimedia.org/api/rest_v1/media/math/render/svg/c715ad94a83af2bbcc5eacc1ad48508e419dba96) Ecuación 5

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+out_%7Bo1%7D%7D%7B%5Cpartial+net_%7Bo1%7D%7D+%3D+out_%7Bo1%7D%281+-+out_%7Bo1%7D%29+%3D+0.75136507%281+-+0.75136507%29+%3D+0.186815602&bg=ffffff&fg=404040&s=0&zoom=2)

In [ ]:
#por lo tanto:

delta_o1_wr_net_o1=out_o1*(1-out_o1)
print(delta_o1_wr_net_o1)

0.18681560180895948


#### 3.1.3 Componente 3: cómo cambia el valor neto de o1 cuando cambia w5

Es simplemente el valor de la neurona h1:

![texto alternativo](https://s0.wp.com/latex.php?latex=net_%7Bo1%7D+%3D+w_5+%2A+out_%7Bh1%7D+%2B+w_6+%2A+out_%7Bh2%7D+%2B+b_2+%2A+1&bg=ffffff&fg=404040&s=0&zoom=2)

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+net_%7Bo1%7D%7D%7B%5Cpartial+w_%7B5%7D%7D+%3D+1+%2A+out_%7Bh1%7D+%2A+w_5%5E%7B%281+-+1%29%7D+%2B+0+%2B+0+%3D+out_%7Bh1%7D+%3D+0.593269992&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (6)

In [ ]:
#numericamente:

delta_net_o1_wr_w5=out_h1
print(delta_net_o1_wr_w5)

0.5932699921071872


#### 3.1.4 Resultado: ¿cuánto cambia el Error Total cuando cambia w5?

Recordando la Ecuación (3):

Recordando:
![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+w_%7B5%7D%7D+%3D+%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+out_%7Bo1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+out_%7Bo1%7D%7D%7B%5Cpartial+net_%7Bo1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+net_%7Bo1%7D%7D%7B%5Cpartial+w_%7B5%7D%7D&bg=ffffff&fg=404040&s=2&zoom=2) Ecuación 3

In [ ]:
delta_e_total_wr_w5=delta_e_wr_delta_out_o1*delta_o1_wr_net_o1*delta_net_o1_wr_w5
print(delta_e_total_wr_w5)

0.08216704056423078


In [ ]:
## o equivalentemente
e1*out_o1*(1-out_o1)*out_h1


0.08216704056423077

Esta ecuación también se conoce como **delta rule**:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+w_%7B5%7D%7D+%3D+-%28target_%7Bo1%7D+-+out_%7Bo1%7D%29+%2A+out_%7Bo1%7D%281+-+out_%7Bo1%7D%29+%2A+out_%7Bh1%7D&bg=D5E9F6&fg=404040&s=0&zoom=2) Ecuación (7)

Podemos replicar el proceso o usar la delta rule para obtener las derivadas de los parámetros restantes w6, w7 y w8:

In [ ]:
delta_e_total_wr_w6=e1*out_o1*(1-out_o1)*out_h2
delta_e_total_wr_w7=e2*out_o2*(1-out_o2)*out_h1
delta_e_total_wr_w8=e2*out_o2*(1-out_o2)*out_h2

print(delta_e_total_wr_w6,delta_e_total_wr_w7,delta_e_total_wr_w8)

0.08266762784753325 -0.02260254047747507 -0.022740242215978222


### 3.2 Actualizando los Parámetros de la Capa de Salida

Conociendo el gradiente (la matriz de todas las derivadas parciales), calculamos los nuevos pesos siguiendo la lógica del gradiente descendente: nos movemos en la dirección opuesta al gradiente para minimizar el error.

Por ejemplo, para actualizar w5:

![texto alternativo](https://s0.wp.com/latex.php?latex=w_5%5E%7B%2B%7D+%3D+w_5+-+%5Ceta+%2A+%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+w_%7B5%7D%7D+%3D+0.4+-+0.5+%2A+0.082167041+%3D+0.35891648&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (8)

donde **η** (*eta*) es la **tasa de aprendizaje** (*learning rate*); en este caso η = 0.5.

In [ ]:
#numericamente
eta=0.5
w5_new=w5-eta*delta_e_total_wr_w5
print(w5_new)

0.35891647971788465


In [ ]:
#equivalentemente para los otros parámetros:
w6_new=w6-eta*delta_e_total_wr_w6
w7_new=w7-eta*delta_e_total_wr_w7
w8_new=w8-eta*delta_e_total_wr_w8

print(w6_new,w7_new,w8_new)

0.4086661860762334 0.5113012702387375 0.5613701211079891


## 4. Actualización de los Parámetros de la Capa Oculta

Procedamos a estimar el efecto de los parámetros que están más "escondidos" en la red, es decir, aquellos que no se encuentran directamente conectados a las salidas. Por ejemplo, ¿cómo podemos estimar cuánto afecta el parámetro w1 al error total para poder optimizarlo?

**Visualmente:**

![texto alternativo](https://drive.google.com/uc?id=1wlVYusuLRoSmZ-JEwZdjOfN4M532rVSo) Figura 4.

Una vez más ocuparemos el encadenamiento de efectos, o **regla de la cadena**. En la Figura 4 podemos ver que: el Error Total depende de la salida de o1 y o2; estas dependen del output de h1; ese output depende del valor neto de h1; y ese valor neto depende de w1.

Matemáticamente:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+w_%7B1%7D%7D+%3D+%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+out_%7Bh1%7D%7D%7B%5Cpartial+net_%7Bh1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+net_%7Bh1%7D%7D%7B%5Cpartial+w_%7B1%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (9)

La Ecuación 9 está compuesta por 3 componentes principales, cada uno "encadenado" a resultados de capas posteriores. El primer término puede descomponerse en dos sub-componentes, uno por cada error (Ecuación 10):

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%3D+%5Cfrac%7B%5Cpartial+E_%7Bo1%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%2B+%5Cfrac%7B%5Cpartial+E_%7Bo2%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (10)

### 4.1 Estimación del Primer Término de la Ecuación 10

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Bo1%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%3D+%5Cfrac%7B%5Cpartial+E_%7Bo1%7D%7D%7B%5Cpartial+net_%7Bo1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+net_%7Bo1%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (11)

#### 4.1.1 Primer componente: ya conocemos su estructura y valores numéricos

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Bo1%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (12)

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Bo1%7D%7D%7B%5Cpartial+net_%7Bo1%7D%7D+%3D+%5Cfrac%7B%5Cpartial+E_%7Bo1%7D%7D%7B%5Cpartial+out_%7Bo1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+out_%7Bo1%7D%7D%7B%5Cpartial+net_%7Bo1%7D%7D+%3D+0.74136507+%2A+0.186815602+%3D+0.138498562&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (13)

In [ ]:
delta_e_o1_wr_delta_out_o1=-(o1-out_o1)
print(delta_e_o1_wr_delta_out_o1)

0.7413650695523157


In [ ]:
delta_o1_wr_net_o1=out_o1*(1-out_o1)
print(delta_o1_wr_net_o1)

0.18681560180895948


In [ ]:
# valor estimado:
delta_eo1_wr_net_o1=delta_e_o1_wr_delta_out_o1*delta_o1_wr_net_o1
print(delta_eo1_wr_net_o1)

0.13849856162855698


#### 4.1.2 Segundo componente

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+net_%7Bo1%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (14)

Podemos estimar esta derivada como:

![texto alternativo](https://s0.wp.com/latex.php?latex=net_%7Bo1%7D+%3D+w_5+%2A+out_%7Bh1%7D+%2B+w_6+%2A+out_%7Bh2%7D+%2B+b_2+%2A+1&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (15)

Lo que nos da como resultado simplemente w5:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+net_%7Bo1%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%3D+w_5+%3D+0.40&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (16)

In [ ]:
delta_net_o1_wr_out_h1=w5
print(delta_net_o1_wr_out_h1)

0.4


Con todos los componentes podemos estimar el primer término de la Ecuación (10):

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Bo1%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%3D+%5Cfrac%7B%5Cpartial+E_%7Bo1%7D%7D%7B%5Cpartial+net_%7Bo1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+net_%7Bo1%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%3D+0.138498562+%2A+0.40+%3D+0.055399425&bg=ffffff&fg=404040&s=0&zoom=2)

In [ ]:
delta_e_o1_wr_out_h1=delta_eo1_wr_net_o1*delta_net_o1_wr_out_h1
print(delta_e_o1_wr_out_h1)

0.05539942465142279


### 4.2 Estimación del Segundo Término de la Ecuación 10

Replicamos la misma lógica para estimar el segundo término:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Bo2%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2)

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Bo2%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%3D+-0.019049119&bg=ffffff&fg=404040&s=0&zoom=2)

In [ ]:
# Estimando el segundo componente de la ecuación 10:
delta_e_o2_wr_delta_out_o2=-(o2-out_o2)
print("delta_e_o2_wr_delta_out_o2")
print(delta_e_o2_wr_delta_out_o2)


delta_o2_wr_net_o2=out_o2*(1-out_o2)
print("delta_o2_wr_net_o2")
print(delta_o2_wr_net_o2)


delta_eo2_wr_net_o2=delta_e_o2_wr_delta_out_o2*delta_o2_wr_net_o2
print("delta_eo2_wr_net_o2")

print(delta_eo2_wr_net_o2)

delta_net_o2_wr_out_h1=w7  # notar que en este caso la derivada nos da w7
print("delta_net_o2_wr_out_h1")

print(delta_net_o2_wr_out_h1)

delta_e_o2_wr_out_h1=delta_eo2_wr_net_o2*delta_net_o2_wr_out_h1
print("delta_e_o2_wr_out_h1")
print(delta_e_o2_wr_out_h1)

delta_e_o2_wr_delta_out_o2
-0.21707153467853746
delta_o2_wr_net_o2
0.17551005281727122
delta_eo2_wr_net_o2
-0.03809823651655623
delta_net_o2_wr_out_h1
0.5
delta_e_o2_wr_out_h1
-0.019049118258278114


El resultado de la Ecuación (10) es:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%3D+%5Cfrac%7B%5Cpartial+E_%7Bo1%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%2B+%5Cfrac%7B%5Cpartial+E_%7Bo2%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%3D+0.055399425+%2B+-0.019049119+%3D+0.036350306&bg=ffffff&fg=404040&s=0&zoom=2)

In [ ]:
# El resultado de la ecuación 10 es:

delta_e_total_delta_out_h1=delta_e_o1_wr_out_h1+delta_e_o2_wr_out_h1
print("delta_e_total_delta_out_h1")
print(delta_e_total_delta_out_h1)


delta_e_total_delta_out_h1
0.03635030639314468


### 4.3 Cálculo Final del Gradiente de w1

La Ecuación (10) representa solo el primer componente de la Ecuación (9), la que nos dará el efecto total de w1 sobre el error total:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+w_%7B1%7D%7D+%3D+%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+out_%7Bh1%7D%7D%7B%5Cpartial+net_%7Bh1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+net_%7Bh1%7D%7D%7B%5Cpartial+w_%7B1%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2) Ecuación (9)

Estimamos ahora los otros dos componentes siguiendo la misma lógica de la sección anterior.

Para el componente:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+out_%7Bh1%7D%7D%7B%5Cpartial+net_%7Bh1%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2)

sabemos que:

![texto alternativo](https://s0.wp.com/latex.php?latex=out_%7Bh1%7D+%3D+%5Cfrac%7B1%7D%7B1%2Be%5E%7B-net_%7Bh1%7D%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2)

y que su derivada respecto de la entrada neta es:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+out_%7Bh1%7D%7D%7B%5Cpartial+net_%7Bh1%7D%7D+%3D+out_%7Bh1%7D%281+-+out_%7Bh1%7D%29+%3D+0.59326999%281+-+0.59326999+%29+%3D+0.241300709&bg=ffffff&fg=404040&s=0&zoom=2)

In [ ]:
delta_out_h1_wr_delta_net_h1=out_h1*(1-out_h1)
print("delta_out_h1_wr_delta_net_h1")
print(delta_out_h1_wr_delta_net_h1)

delta_out_h1_wr_delta_net_h1
0.24130070857232525


Para el componente:

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+net_%7Bh1%7D%7D%7B%5Cpartial+w%7D&bg=ffffff&fg=404040&s=0&zoom=2)

calculamos la derivada del valor neto respecto al w correspondiente (en este caso w1):

![texto alternativo](https://s0.wp.com/latex.php?latex=net_%7Bh1%7D+%3D+w_1+%2A+i_1+%2B+w_3+%2A+i_2+%2B+b_1+%2A+1&bg=ffffff&fg=404040&s=0&zoom=2)

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+net_%7Bh1%7D%7D%7B%5Cpartial+w_1%7D+%3D+i_1+%3D+0.05&bg=ffffff&fg=404040&s=0&zoom=2)

In [ ]:
delta_net_h1_wr_w1=i1
print("delta_net_h1_wr_w1")
print(delta_net_h1_wr_w1)

delta_net_h1_wr_w1
0.05


Ahora ya contamos con todos los componentes de la Ecuación (9):

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+w_%7B1%7D%7D+%3D+%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+out_%7Bh1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+out_%7Bh1%7D%7D%7B%5Cpartial+net_%7Bh1%7D%7D+%2A+%5Cfrac%7B%5Cpartial+net_%7Bh1%7D%7D%7B%5Cpartial+w_%7B1%7D%7D&bg=ffffff&fg=404040&s=0&zoom=2)

![texto alternativo](https://s0.wp.com/latex.php?latex=%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+w_%7B1%7D%7D+%3D+0.036350306+%2A+0.241300709+%2A+0.05+%3D+0.000438568&bg=ffffff&fg=404040&s=0&zoom=2)

In [ ]:
delta_e_total_wr_delta_w1=delta_e_total_delta_out_h1*delta_out_h1_wr_delta_net_h1*delta_net_h1_wr_w1
print("delta_e_total_wr_delta_w1")
print(delta_e_total_wr_delta_w1)


delta_e_total_wr_delta_w1
0.00043856773447434685


### 4.4 Actualizando los Parámetros de la Capa Oculta

Habiendo estimado la derivada del error total respecto de w1 (Ecuación 9), usamos ese gradiente para actualizar el parámetro hacia un valor que disminuya el error. El nuevo valor de w1 será:

![texto alternativo](https://s0.wp.com/latex.php?latex=w_1%5E%7B%2B%7D+%3D+w_1+-+%5Ceta+%2A+%5Cfrac%7B%5Cpartial+E_%7Btotal%7D%7D%7B%5Cpartial+w_%7B1%7D%7D+%3D+0.15+-+0.5+%2A+0.000438568+%3D+0.149780716&bg=ffffff&fg=404040&s=0&zoom=2)

In [ ]:
w1_new=w1-eta*delta_e_total_wr_delta_w1
print("w1_new")
print(w1_new)


w1_new
0.1497807161327628


Repitiendo este procedimiento para w2, w3 y w4 obtenemos:

![texto alternativo](https://s0.wp.com/latex.php?latex=w_2%5E%7B%2B%7D+%3D+0.19956143&bg=ffffff&fg=404040&s=0&zoom=2)

![texto alternativo](https://s0.wp.com/latex.php?latex=w_3%5E%7B%2B%7D+%3D+0.24975114&bg=ffffff&fg=404040&s=0&zoom=2)

![texto alternativo](https://s0.wp.com/latex.php?latex=w_4%5E%7B%2B%7D+%3D+0.29950229&bg=ffffff&fg=404040&s=0&zoom=2)

### Ejercicio Propuesto

Verifique que los nuevos valores de w2, w3 y w4 son los indicados en la celda anterior.

## 5. Nueva Iteración con los Parámetros Actualizados

Si ahora ocupamos los nuevos valores de los parámetros obtendremos:

In [ ]:
# Propagando los datos hacia adelante ->

# Calculamos los valores netos de la neuronas de la capa oculta (netos, es decir, antes de aplicarles la función de activación logística)
# podemos ver que es solo una suma ponderada de los inputs a los que se les suma un término constante o "bias":

w1=w1_new
w2=0.19956143
w3=0.24975114
w4=0.29950229
w5=w5_new
w6=w6_new
w7=w8_new

net_h1=i1*w1+i2*w2+b1
net_h2=i1*w3+i2*w4+b1

print('net_h1='+ str(net_h1))
print('net_h2='+str(net_h2))

net_h1=0.37744517880663814
net_h2=0.392437786


In [ ]:
# necesitamos aplicar la función de activación logística (u otra).

# ahora la aplico a cada una de las neuronas de la capa oculta

out_h1=logistic_func(net_h1)
out_h2=logistic_func(net_h2)

print('out_h1='+ str(out_h1))
print('out_h2='+ str(out_h2))


out_h1=0.5932567636467482
out_h2=0.5968694086464008


In [ ]:
# calculamos el valor neto ocupando los pesos respectivos. notar que el input de las neuronas de la capa de salida o1 y o2,
#es el output de las neuronas de la capa oculta

net_o1=out_h1*w5+out_h2*w6+b2
net_o2=out_h1*w7+out_h2*w8+b2

print('net_o1='+ str(net_o1))
print('net_o2='+ str(net_o2))

net_o1=1.0568499739940174
net_o2=1.2613147960120292


In [ ]:
# nuevamente, a estos valores netos le aplicamos la función de activación Ecuación 1.
out_o1=logistic_func(net_o1)
out_o2=logistic_func(net_o2)

print('out_o1='+ str(out_o1))
print('out_o2='+ str(out_o2))

out_o1=0.7420881111887568
out_o2=0.7792523595555513


In [ ]:
# Calculamos los errores

e1=out_o1-o1
e2=out_o2-o2

print(e1,e2)

0.7320881111887568 -0.21074764044444871


In [ ]:
loss_e1=0.5*(e1**2)
loss_e2=0.5*(e2**2)

print(loss_e1,loss_e2)

0.26797650127196077 0.022207283976451317


In [ ]:
total_error=loss_e1+loss_e2
print(total_error)

0.2901837852484121


Podemos notar que el error total en esta iteración (con η = 0.5) disminuyó considerablemente desde el valor inicial **0.298371109** hasta **0.2901837852484121**. Si repitiéramos este proceso muchas veces lograríamos disminuir el error de manera consistente, en algunos casos llegando incluso a Error Total = 0.

Si la red contara con muchas más capas ocultas, se seguiría repitiendo esta misma lógica hasta lograr optimizar todos sus parámetros. La misma lógica puede extenderse también para optimizar los parámetros b1 y b2 (*bias*).